# 23 · Spans, not vibes

## Goal

Turn on environment-level OTel span export to Application Insights, find
the fleet in the Agents (Preview) view, run a KQL query over tool-call
latency and token/credit attribution, and replay one failing golden case
deterministically from its span.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.checkpoint import checkpoint
checkpoint(
    name="Environment-level OTel span export enabled (PREVIEW, since 15 Jul 2026)",
    probe=lambda: input("Enabled in PPAC for this environment, pointed at App Insights? (y/n): ") == "y",
    remediation="Power Platform admin center > environment > Settings > Product > Features > enable OpenTelemetry span export, set the App Insights connection string.",
)


## Concept

**Finding #13 makes this notebook stronger than originally planned.**
Environment-level span export to App Insights is a PPAC toggle (PREVIEW
since 15 Jul 2026), and App Insights now has an **Agents (Preview)** view
that unifies Foundry, Copilot Studio, and third-party agent traces in one
place — so the connected-agent fleet from `20`-`22` shows up as one
coherent trace tree, tool call by tool call, model by model, rather than
five separate logs you'd have to correlate by hand.

The habit this notebook builds: when a golden case fails, don't guess —
find its span, read the actual tool calls and latencies, and replay just
that one case. That's strictly better than re-running the whole suite and
hoping the failure repeats.


## Build


### KQL: tool-call latency and credit attribution across the fleet


In [ ]:
kql_latency_by_tool = '''
AgentTraces
| where TimeGenerated > ago(1h)
| where AgentName in ("contract-renewal-desk", "drafting-specialist", "critic-reviewer", "extraction-agent", "routing-agent")
| summarize p50=percentile(DurationMs, 50), p95=percentile(DurationMs, 95), count() by ToolName, AgentName
| order by p95 desc
'''
print(kql_latency_by_tool)
# Run this in the App Insights Agents (Preview) view or via
# azure-monitor-query's LogsQueryClient — printed here so it's copy-pasteable
# regardless of which surface you're in.


In [ ]:
from azure.monitor.query import LogsQueryClient
from azure.identity import DefaultAzureCredential

logs_client = LogsQueryClient(DefaultAzureCredential())
response = logs_client.query_workspace(
    workspace_id="$LOG_ANALYTICS_WORKSPACE_ID",
    query=kql_latency_by_tool,
    timespan=None,
)
for table in response.tables:
    for row in table.rows[:10]:
        print(row)


### Sensitive-property logging — an explicit decision, not a default


In [ ]:
# Span export can include request/response payloads. Deciding to log full
# supplier pricing data into App Insights is a privacy decision, not a
# technical default — make it explicitly and record it.
LOG_FULL_PAYLOADS = False  # deliberate choice: span metadata only, not response bodies, for this workshop tenant
print(f"sensitive-property logging: {'ON — payloads included' if LOG_FULL_PAYLOADS else 'OFF — metadata/latency only'}")


## Verify

Same harness, same golden set, every notebook.


Replay one failing case deterministically, from its span.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

suite = run_suite(client, cases=load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)
failing = [r for r in suite.results if not r.passed]
if failing:
    case_id = failing[0].case_id
    print(f"replaying {case_id} — pull its span from App Insights using its trace/span ID from the client response, not a guess")
else:
    print("nothing failing to replay this run — re-run 06's ungrounded-policy case with a deliberately bad instructions.md edit to see the replay flow")


## Cost


In [ ]:
meter.report_cost("23", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="regression run + span export (App Insights ingestion billed separately from Copilot Credits)")


## Teardown


In [ ]:
print("No teardown — span export stays on; it's the observability floor for 24-25.")
